# Coastal & Wetland Mapping
**NB 19** | Louisiana Coastal Wetlands, USA | Sentinel-2 | SegFormer

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import pathlib

import numpy as np

import pygeovision as pgv
from pygeovision.inference.tiled import TiledInference
from pygeovision.models import get_model

client = pgv.PyGeoVision()
print(client)


In [ ]:
BBOX       = (-91.5, 29.3, -91.0, 29.7)
DATE_RANGE = ('2024-03-01', '2024-09-30')
PROVIDERS  = ['planetary_computer']

DATA_DIR = pathlib.Path('./results/notebook')
DATA_DIR.mkdir(parents=True, exist_ok=True)

results = client.search(
    bbox            = BBOX,
    date_range      = DATE_RANGE,
    providers       = PROVIDERS,
    cloud_cover_max = 20,
)
print(f"Found {len(results)} scenes")
for r in results[:5]:
    print(f"  {r.provider:<22} {r.datetime[:10]}  cloud={r.cloud_cover:.1f}%  {r.id[:40]}")


In [ ]:
BANDS = ['B02', 'B03', 'B04', 'B08', 'B11', 'B12']

downloads = client.download(
    results[:1],
    output_dir   = str(DATA_DIR),
    bands        = BANDS,
    post_process = ['reproject:EPSG:32615', 'cog'],
)
scene_path = downloads[0].path if downloads and downloads[0].success else None
scl_cands  = list(DATA_DIR.rglob('*SCL*.tif')) + list(DATA_DIR.rglob('*scl*.tif'))
scl_path   = str(scl_cands[0]) if scl_cands else None
if scene_path:
    d = downloads[0]
    print(f"Downloaded   : {scene_path}")
    print(f"Size         : {d.bytes_downloaded/1024/1024:.1f} MB")
    print(f"SCL mask     : {scl_path}")
else:
    print("Download failed or scene unavailable — check provider availability")


In [ ]:
if scene_path:
    raw_report = client.validator.validate(
        str(scene_path),
        required_bands = 6,
        value_range    = (0, 65535),
    )
    print("Raw data validation:")
    print(raw_report.summary())
    if not raw_report.passed:
        print("WARNING: Issues found — preprocessing will auto-fix")


In [ ]:
PREPROCESSED = DATA_DIR / 'louisiana_preprocessed.tif'

if scene_path:
    ready = client.prepare_for_ai(
        str(scene_path),
        stack_bands  = BANDS,
        bbox         = BBOX,
        scl_path         = scl_path,
        scl_keep_classes = [4,5,6],
        normalise    = 'scale_factor',
        scale_factor = 10000.0,
        model_type   = 'segmentation',
        output_path  = str(PREPROCESSED),
    )
    print("Preprocessing steps :", ready['steps'])
    print("Output shape (C,H,W):", ready['shape'])
    print("Resolution          :", ready['resolution_m'], "m")
    print("Validation          :", "PASSED" if ready['report'] and ready['report'].passed else "FIXED (auto)")
    arr = ready['array']
    print(f"Value range         : {arr.min():.4f} → {arr.max():.4f}")
    print(f"NaN count           : {np.isnan(arr).sum()}")
else:
    print("No scene available — cannot preprocess")
    ready = None


In [ ]:
# Wetland indices: NDWI, MNDWI, LSWI, BSI
if PREPROCESSED.exists():
    idx_dir = DATA_DIR/'indices'; idx_dir.mkdir(exist_ok=True)
    ndwi  = client.indices.ndwi( str(PREPROCESSED), green_band=2, nir_band=4, output_path=str(idx_dir/'ndwi.tif'))
    mndwi = client.indices.mndwi(str(PREPROCESSED), green_band=2, swir1_band=5,output_path=str(idx_dir/'mndwi.tif'))
    lswi  = client.indices.lswi( str(PREPROCESSED), nir_band=4,  swir1_band=5,output_path=str(idx_dir/'lswi.tif'))
    import rasterio
    for name, p in [('NDWI',ndwi),('MNDWI',mndwi),('LSWI',lswi)]:
        with rasterio.open(p) as src: a = src.read(1)
        water_pct = (a > 0.1).mean() * 100
        print(f"{name}: water={water_pct:.1f}%  mean={a.mean():.3f}")


In [ ]:
PREDICTION = DATA_DIR / 'wetland_raw.tif'

if ready and PREPROCESSED.exists():
    model = get_model('segformer-b2', num_classes=4, pretrained=True)
    inf   = TiledInference(model, chip_size=512, overlap=64, blend_mode='gaussian')
    res   = inf.infer(str(PREPROCESSED), str(PREDICTION))
    print(f"Inference: {res.get('n_chips')} chips  {res.get('duration_seconds',0):.1f}s  "
          f"{res.get('chips_per_second',0):.1f} chips/s")
else:
    print("Skipping inference — no preprocessed scene")


In [ ]:
if PREDICTION.exists():
    pred_report = client.validator.validate(str(PREDICTION))
    print("Prediction validation:", "PASSED" if pred_report.passed else f"FIXED — {pred_report.errors}")
    print(pred_report.summary())


In [ ]:
SIEVED = DATA_DIR/'wetland_sieved.tif'; VECTOR = DATA_DIR/'wetland_polygons.geojson'
COG_OUT = DATA_DIR/'wetland_cog.tif'
CLASS_NAMES = {0:'Open water', 1:'Marsh/wetland', 2:'Upland', 3:'Mudflat/bare'}
if PREDICTION.exists():
    client.postprocess.sieve_filter(str(PREDICTION), min_pixels=30, output_path=str(SIEVED))
    client.postprocess.vectorise(str(SIEVED), str(VECTOR), min_area_m2=500.0, simplify_tolerance=1.0)
    stats = client.postprocess.class_statistics(str(SIEVED))
    client.postprocess.to_cog(str(SIEVED), str(COG_OUT))
    print(f"{'Cls':<4} {'Habitat':<18} {'Area(ha)':>10} {'Cover':>8}")
    print('-'*44)
    for cls, name in CLASS_NAMES.items():
        inf = stats.get(cls, {})
        print(f"{cls:<4} {name:<18} {inf.get('area_ha',0):>10.1f} {inf.get('pct',0):>7.1f}%")


**NB19 — Coastal Wetland** complete.
- Study area: Louisiana USA
- Sensor: Sentinel-2 6-band
- Model: SegFormer-B2 4-class
- Pipeline: search → download → validate_raw → preprocess → validate_clean → model → postprocess → COG